In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import glob
from xgrads import open_CtlDataset
from pathlib import Path
import netCDF4

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output

ncl_cmap = LinearSegmentedColormap.from_list(
    "BlueWhiteOrangeRed",
    ["#2166ac", "#67a9cf", "#ffffff", "#fdae61", "#b2182b"],
    N=256
)


plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 160)

print("Python OK")

Python OK


In [2]:
BASE_DIR = Path.cwd()

NEW_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_102"
OLD_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_101"
print("NEW_DIR:", NEW_DIR, NEW_DIR.exists())
print("OLD_DIR:", OLD_DIR, OLD_DIR.exists())

# print("\nNew files:")
# if NEW_DIR.exists():
#     for p in sorted(NEW_DIR.glob("*.grd")):
#         print(f"{p.name:25s} {p.stat().st_size / 1024**2:.2f} MB")
# else:
#     print("NEW_DIR does not exist")

# print("\nOld files:")
# if OLD_DIR.exists():
#     for p in sorted(OLD_DIR.glob("*.grd")):
#          print(f"{p.name:25s} {p.stat().st_size / 1024**2:.2f} MB")
# else:
#     print("OLD_DIR does not exist")   

SPEEDY_VARIABLE = "TEMP0"
ACCESS_VARIABLE = "tas"   

# Output directory
TAS_OUT_DIR = (Path.cwd() / ".." / ".." / "SPEEDY_access" / "access_forcing").resolve()
TAS_OUT_DIR.mkdir(parents=True, exist_ok=True)
WRITE_ONE_FILE_PER_YEAR = True
# Keep the SPEEDY grid untouched until the JRA-55 metadata/grid are inspected.
SHIFT_LONGITUDE_TO_MINUS180_180 = False
SORT_LATITUDE_NORTH_TO_SOUTH = False
print("Output directory:", TAS_OUT_DIR)


NEW_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_102 True
OLD_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_101 True
Output directory: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/SPEEDY_access/access_forcing


In [3]:
for ctl in Path(NEW_DIR).glob("attm102.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))
ds

attm102.ctl


<xarray.Dataset> Size: 17GB
Dimensions:  (time: 8760, lev: 8, lat: 48, lon: 96)
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lev      (lev) float64 64B 925.0 850.0 700.0 500.0 300.0 200.0 100.0 30.0
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables: (12/43)
    GH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    TEMP     (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    U        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    V        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    Q        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    RH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    ...       ...
    SHF      (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    LSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SLRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SNOW     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
Attributes:
    comment:  geopotential height               [m]
    storage:  99
    title:    Means/variances
    undef:    9.999e+19
    pdef:     None

In [4]:
# =========================
# 2. Inspect source ST


if SPEEDY_VARIABLE not in ds:
    raise KeyError(
        f"{SPEEDY_VARIABLE!r} is absent from the SPEEDY dataset. "
        f"Available variables: {list(ds.data_vars)}"
    )

st = ds[SPEEDY_VARIABLE]

print(st)
print("\nDimensions:", st.dims)
print("Shape:", st.shape)
print("Dtype:", st.dtype)
print("Attributes:", st.attrs)

dt_hours = np.diff(ds.time.values) / np.timedelta64(1, "h")
print("\nUnique output intervals [hours]:", np.unique(dt_hours))

print(
    "\nST range [K]:",
    float(st.min().compute()),
    "to",
    float(st.max().compute()),
)

print(
    "ST global mean [K]:",
    float(st.mean(skipna=True).compute()),
)


<xarray.DataArray 'TEMP0' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Attributes:
    comment:  near-surface air temperature   [degK]
    storage:  99

Dimensions: ('time', 'lat', 'lon')
Shape: (8760, 48, 96)
Dtype: >f4
Attributes: {'comment': 'near-surface air temperature   [degK]', 'storage': '99'}

Unique output intervals [hours]: [3.]

ST range [K]: 208.4306182861328 to 315.99871826171875
ST global mean [K]: 279.9093322753906


In [5]:
# =========================
# 3. Build provisional tas

# Do NOT subtract 273.15 here:
# ACCESS-style air temperature forcing is normally stored in kelvin.
tas = ds[SPEEDY_VARIABLE].rename(ACCESS_VARIABLE).astype("float32")

# Optional grid transformations, disabled by default.
if SHIFT_LONGITUDE_TO_MINUS180_180:
    tas = tas.assign_coords(
        lon=(((tas.lon + 180.0) % 360.0) - 180.0)
    ).sortby("lon")

if SORT_LATITUDE_NORTH_TO_SOUTH:
    tas = tas.sortby("lat", ascending=False)

# Provisional CF-style metadata.
# These attributes will be made identical to JRA-55 after its metadata are supplied.
tas.attrs = {
    "standard_name": "air_temperature",
    "long_name": "Near-surface air temperature",
    "units": "K",
    "source_variable": SPEEDY_VARIABLE,
    "source_model": "SPEEDY",
    "mapping_note": "Provisional mapping SPEEDY ST to ACCESS-OM2 tas",
}

# Coordinate metadata; names remain lat/lon for now.
if "lat" in tas.coords:
    tas["lat"].attrs.update({
        "standard_name": "latitude",
        "long_name": "latitude",
        "units": "degrees_north",
        "axis": "Y",
    })

if "lon" in tas.coords:
    tas["lon"].attrs.update({
        "standard_name": "longitude",
        "long_name": "longitude",
        "units": "degrees_east",
        "axis": "X",
    })

tas["time"].attrs.update({
    "standard_name": "time",
    "long_name": "time",
    "axis": "T",
})

tas_ds = tas.to_dataset()

tas_ds.attrs = {
    "title": "SPEEDY forcing for ACCESS-OM2",
    "institution": "",
    "source": "SPEEDY model output",
    "history": "Created from SPEEDY ST and renamed to tas",
    "comment": (
        "Provisional NetCDF export. Metadata, coordinate names and grid "
        "conventions must be checked against the target JRA-55 forcing files."
    ),
}

tas_ds

<xarray.Dataset> Size: 162MB
Dimensions:  (time: 8760, lat: 48, lon: 96)
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables:
    tas      (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
Attributes:
    title:        SPEEDY forcing for ACCESS-OM2
    institution:  
    source:       SPEEDY model output
    history:      Created from SPEEDY ST and renamed to tas
    comment:      Provisional NetCDF export. Metadata, coordinate names and g...

In [6]:
# =========================
# 4. Basic validation

required_dims = {"time", "lat", "lon"}

if set(tas.dims) != required_dims:
    raise ValueError(
        f"Expected tas dimensions {required_dims}, got {tas.dims}"
    )

if tas.attrs.get("units") != "K":
    raise ValueError("tas must currently be stored in kelvin")

if not np.issubdtype(tas.dtype, np.floating):
    raise TypeError(f"tas must be floating point, got {tas.dtype}")

# Check that no invalid SPEEDY undef values remain.
undef = ds.attrs.get("undef", 9.999e19)
invalid_count = int((np.abs(tas) >= abs(undef) * 0.9).sum().compute())

if invalid_count:
    raise ValueError(
        f"Found {invalid_count} values close to SPEEDY undef={undef}"
    )

# Broad physical sanity check only; not a scientific validation.
tas_min = float(tas.min().compute())
tas_max = float(tas.max().compute())

if not (150.0 < tas_min < 350.0):
    print(f"WARNING: unusual minimum tas={tas_min:.3f} K")

if not (200.0 < tas_max < 400.0):
    print(f"WARNING: unusual maximum tas={tas_max:.3f} K")

print("Validation passed")
print(f"tas minimum: {tas_min:.3f} K")
print(f"tas maximum: {tas_max:.3f} K")
print("time:", tas.time.values[0], "to", tas.time.values[-1])
print("grid:", tas.sizes["lat"], "x", tas.sizes["lon"])


Validation passed
tas minimum: 208.431 K
tas maximum: 315.999 K
time: 1989-01-01T00:00:00.000000000 to 1991-12-31T21:00:00.000000000
grid: 48 x 96


In [7]:
# =========================
# 5. NetCDF encoding

# Chunking is selected for time-series forcing access:
# one time record per chunk, complete horizontal field.
encoding = {
    ACCESS_VARIABLE: {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
        "_FillValue": np.float32(1.0e20),
        "chunksizes": (
            1,
            tas.sizes["lat"],
            tas.sizes["lon"],
        ),
    },
    "time": {
        # Provisional time encoding. It will be aligned exactly with JRA-55.
        "units": "hours since 1900-01-01 00:00:00",
        "calendar": "365_day",
    },
    "lat": {
        "dtype": "float64",
        "_FillValue": None,
    },
    "lon": {
        "dtype": "float32",
        "_FillValue": None,
    },
}

encoding


{'tas': {'dtype': 'float32',
  'zlib': True,
  'complevel': 4,
  'shuffle': True,
  '_FillValue': np.float32(1e+20),
  'chunksizes': (1, 48, 96)},
 'time': {'units': 'hours since 1900-01-01 00:00:00', 'calendar': '365_day'},
 'lat': {'dtype': 'float64', '_FillValue': None},
 'lon': {'dtype': 'float32', '_FillValue': None}}

In [8]:
# =========================
# 6. Write NetCDF files

written_files = []

if WRITE_ONE_FILE_PER_YEAR:
    years = np.unique(tas_ds.time.dt.year.values)

    for year in years:
        yearly = tas_ds.sel(time=str(int(year)))

        output_file = TAS_OUT_DIR / f"tas_SPEEDY_{int(year)}.nc"

        yearly.to_netcdf(
            output_file,
            mode="w",
            format="NETCDF4",
            engine="netcdf4",
            unlimited_dims=["time"],
            encoding=encoding,
        )

        written_files.append(output_file)
        print(
            f"Wrote {output_file.name}: "
            f"{yearly.sizes['time']} records, "
            f"{output_file.stat().st_size / 1024**2:.2f} MB"
        )
else:
    output_file = TAS_OUT_DIR / "tas_SPEEDY_all_years.nc"

    tas_ds.to_netcdf(
        output_file,
        mode="w",
        format="NETCDF4",
        engine="netcdf4",
        unlimited_dims=["time"],
        encoding=encoding,
    )

    written_files.append(output_file)
    print(
        f"Wrote {output_file.name}: "
        f"{tas_ds.sizes['time']} records, "
        f"{output_file.stat().st_size / 1024**2:.2f} MB"
    )

written_files


Wrote tas_SPEEDY_1989.nc: 2920 records, 31.94 MB
Wrote tas_SPEEDY_1990.nc: 2920 records, 31.99 MB
Wrote tas_SPEEDY_1991.nc: 2920 records, 31.99 MB


[PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/SPEEDY_access/access_forcing/tas_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/SPEEDY_access/access_forcing/tas_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/SPEEDY_access/access_forcing/tas_SPEEDY_1991.nc')]

In [9]:
# =========================
# 7. Reopen and verify output

if not written_files:
    raise RuntimeError("No NetCDF files were written")

check_file = written_files[0]

with xr.open_dataset(check_file, decode_times=True) as check:
    print(check)
    print("\nVariable attributes:")
    print(check[ACCESS_VARIABLE].attrs)

    print("\nEncoding:")
    print(check[ACCESS_VARIABLE].encoding)

    print("\nTime:")
    print(check.time.values[0], "to", check.time.values[-1])

    print(
        "\nRange [K]:",
        float(check[ACCESS_VARIABLE].min()),
        "to",
        float(check[ACCESS_VARIABLE].max()),
    )

    # Compare first exported field against the source field.
    source_first = tas.isel(time=0).compute()
    output_first = check[ACCESS_VARIABLE].isel(time=0).load()

    max_abs_difference = float(
        np.abs(source_first - output_first).max()
    )

    print(
        "Maximum absolute difference after NetCDF round trip:",
        max_abs_difference,
    )

    if max_abs_difference != 0.0:
        print(
            "NOTE: a tiny difference may arise only if dtype or packing is changed."
        )


<xarray.Dataset> Size: 54MB
Dimensions:  (time: 2920, lat: 48, lon: 96)
Coordinates:
  * time     (time) object 23kB 1989-01-01 00:00:00 ... 1989-12-31 21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables:
    tas      (time, lat, lon) float32 54MB ...
Attributes:
    title:        SPEEDY forcing for ACCESS-OM2
    institution:  
    source:       SPEEDY model output
    history:      Created from SPEEDY ST and renamed to tas
    comment:      Provisional NetCDF export. Metadata, coordinate names and g...

Variable attributes:
{'standard_name': 'air_temperature', 'long_name': 'Near-surface air temperature', 'units': 'K', 'source_variable': 'TEMP0', 'source_model': 'SPEEDY', 'mapping_note': 'Provisional mapping SPEEDY ST to ACCESS-OM2 tas'}

Encoding:
{'dtype': dtype('float32'), 'zlib': True, 'szip': False, 'zstd': False, 'bzip2': False, 'blosc': False, 'shuffle':